# Liver Regeneration Analysis

In [1]:
import numpy as np
import tifffile
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path

import napari
import torch
import trimesh
import skimage 

from scipy.spatial import distance
from scipy import ndimage as ndi
from numpy.linalg import eigh, norm

from skimage import measure
from skimage.measure import regionprops, marching_cubes
from skimage.segmentation import clear_border, expand_labels
from skimage.morphology import (
    skeletonize,
    remove_small_objects,
    remove_small_holes,
    ball,
    binary_closing,
    binary_opening,
)
import kimimaro



# Check GPU availability for faster processing if needed
print(f"Napari version: {napari.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Napari version: 0.6.6
GPU Available: True
Device: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
DATA_DIR = Path("/home/alejo/Desktop/Master/thesis/Liver_Tissue")

# File names
FILE_BC = "bc.tif"
FILE_CELLS = "cells.tif"
FILE_NUCLEI = "Nuclei.tif"
FILE_SINUSOIDS = "sinusoids.tif"
FILE_VCVP = "VC-VP.tif"

# IDs used in the master label
ID_BILE = 65534
ID_SINU = 65535
ID_BORDER = 65533
ID_CV = 100   
ID_PV = 200   


# HELPERS
def read_volume(path):
    """
    Read a TIFF volume from disk.
    """
    arr = tifffile.imread(path)
    return arr


def get_center_crop_bounds(shape, crop_size_yx):
    """
    Compute central crop bounds for a 3D volume of shape (z, y, x).
    """
    _, dim_y, dim_x = shape

    # If no crop is requested, keep the full field of view
    if crop_size_yx is None:
        return 0, dim_y, 0, dim_x

    if isinstance(crop_size_yx, int):
        crop_y = crop_size_yx
        crop_x = crop_size_yx
    else:
        crop_y, crop_x = crop_size_yx

    crop_y = min(crop_y, dim_y)
    crop_x = min(crop_x, dim_x)

    mid_y = dim_y // 2
    mid_x = dim_x // 2

    y_start = max(0, mid_y - crop_y // 2)
    y_end   = min(dim_y, y_start + crop_y)

    x_start = max(0, mid_x - crop_x // 2)
    x_end   = min(dim_x, x_start + crop_x)

    return y_start, y_end, x_start, x_end


def crop_volume(vol, bounds):
    """
    Apply a (y, x) crop to a 3D volume of shape (z, y, x).
    """
    y_start, y_end, x_start, x_end = bounds
    return vol[:, y_start:y_end, x_start:x_end]


# MAIN LOADER

def load_liver_data(
    data_dir,
    crop_size_yx=None,
    expansion_dist=5,
    include_vessels_in_master=True
):
    """
    Load liver tissue data from separate TIFF files and build a unified master label volume.
    """
    data_dir = Path(data_dir)

    # Read all volumes
    bc_full = read_volume(data_dir / FILE_BC)
    cells_full = read_volume(data_dir / FILE_CELLS)
    nuclei_full = read_volume(data_dir / FILE_NUCLEI)
    sinu_full = read_volume(data_dir / FILE_SINUSOIDS)
    vcvp_full = read_volume(data_dir / FILE_VCVP)

    print("Original volume shapes:")
    print(f"BC: {bc_full.shape}")
    print(f"Cells: {cells_full.shape}")
    print(f"Nuclei: {nuclei_full.shape}")
    print(f"Sinusoids: {sinu_full.shape}")
    print(f"VC-VP: {vcvp_full.shape}")

    # Compute crop bounds and crop all volumes consistently
    ref_shape = cells_full.shape
    bounds = get_center_crop_bounds(ref_shape, crop_size_yx)
    y_start, y_end, x_start, x_end = bounds
    print(f"\nCropping with bounds: Y[{y_start}:{y_end}], X[{x_start}:{x_end}]")

    bc_crop = crop_volume(bc_full, bounds)
    cells_crop = crop_volume(cells_full, bounds)
    nuclei_crop = crop_volume(nuclei_full, bounds)
    sinu_crop = crop_volume(sinu_full, bounds)
    vcvp_crop = crop_volume(vcvp_full, bounds)

    # Expand cell labels
    expanded_cells = expand_labels(cells_crop, distance=expansion_dist)

    # Build the unified master label
    master_labels = expanded_cells.astype(np.uint32).copy()

    # Boolean masks for each structure
    is_bile = bc_crop > 0
    is_sinu = sinu_crop > 0
    is_cv = vcvp_crop == ID_CV
    is_pv = vcvp_crop == ID_PV

    # Insert sinusoidal and bile networks
    master_labels[is_sinu] = ID_SINU
    master_labels[is_bile] = ID_BILE

    # Optionally insert vessels
    if include_vessels_in_master:
        master_labels[is_cv] = ID_CV
        master_labels[is_pv] = ID_PV


    unique_cell_ids = np.unique(expanded_cells)
    n_cells = np.sum(unique_cell_ids > 0)

    print(f"Cropped shape: {expanded_cells.shape}")
    print(f"Unique cell IDs: {n_cells}")
    print(f"Bile voxels: {np.sum(is_bile)}")
    print(f"Sinusoid voxels: {np.sum(is_sinu)}")
    print(f"Central vein voxels: {np.sum(is_cv)}")
    print(f"Portal vein voxels: {np.sum(is_pv)}")

    print("\nUnique values in nuclei crop:")
    print(np.unique(nuclei_crop))

    print("\nUnique values in VC-VP crop:")
    print(np.unique(vcvp_crop))

    return {
        "cells_raw": cells_crop,
        "cells_expanded": expanded_cells,
        "bc_raw": bc_crop,
        "sinusoids_raw": sinu_crop,
        "nuclei_raw": nuclei_crop,
        "vcvp_raw": vcvp_crop,
        "master_labels": master_labels,
        "crop_bounds": bounds
    }

In [4]:
data = load_liver_data(
    data_dir=DATA_DIR,
    crop_size_yx=700,          
    expansion_dist=5,
    include_vessels_in_master=True
)

hepa_img = data["cells_expanded"]
bile_img = data["bc_raw"]
sinu_img = data["sinusoids_raw"]
nuclei_img = data["nuclei_raw"]
vcvp_img = data["vcvp_raw"]
master_labels = data["master_labels"]

print("\nCrop shape:", hepa_img.shape)
unique_cells = np.unique(hepa_img)
print(f"Unique Cell IDs found: {len(unique_cells)-1}")

Original volume shapes:
BC: (307, 1488, 1418)
Cells: (307, 1488, 1418)
Nuclei: (307, 1488, 1418)
Sinusoids: (307, 1488, 1418)
VC-VP: (307, 1488, 1418)

Cropping with bounds: Y[394:1094], X[359:1059]
Cropped shape: (307, 700, 700)
Unique cell IDs: 647
Bile voxels: 5590995
Sinusoid voxels: 23649189
Central vein voxels: 0
Portal vein voxels: 0

Unique values in nuclei crop:
[0 1 4 5 6 9]

Unique values in VC-VP crop:
[0]

Crop shape: (307, 700, 700)
Unique Cell IDs found: 647


In [5]:
def generate_mesh(hepa_img, bile_raw, sinu_raw, smoothing_iters=10):
    """
    Generate meshes
    """
    # Hepatocytes
    hepa_padded = np.pad(hepa_img, pad_width=1, mode='constant', constant_values=0)
    cell_meshes = {}
    unique_ids = np.unique(hepa_img)
    unique_ids = unique_ids[unique_ids > 0] # No background
    
    for cell_id in unique_ids:
        # Mask
        mask = (hepa_padded == cell_id)
        verts, faces, _, _ = measure.marching_cubes(mask, level=0.5) 
        verts -= 1
        # Mesh
        mesh = trimesh.Trimesh(vertices=verts, faces=faces)
        
        # Correction
        if mesh.volume < 0:
            mesh.invert()
            
        # Smooth 
        #trimesh.smoothing.filter_laplacian(mesh, iterations=smoothing_iters)
        
        # Save
        cell_meshes[cell_id] = mesh

    # Networks Bile & Sinusoid
    #vb, fb, _, _ = measure.marching_cubes(bile_raw, level=0.5)
    #mesh_bile = trimesh.Trimesh(vertices=vb, faces=fb)
    #if mesh_bile.volume < 0: mesh_bile.invert()

    
    #vs, fs, _, _ = measure.marching_cubes(sinu_raw, level=0.5)
    #mesh_sinu = trimesh.Trimesh(vertices=vs, faces=fs)
    #if mesh_sinu.volume < 0: mesh_sinu.invert()

    #return cell_meshes, mesh_bile, mesh_sinu
    return cell_meshes

cell_meshes = generate_mesh(hepa_img, bile_img, sinu_img, smoothing_iters=3)

In [13]:
def visualize_meshes(hepa_img, bile_raw, sinu_raw, 
                                   cell_meshes, 
                                   sample_cell_id=None):
    """
    Visualization of meshes.
    """
    viewer = napari.Viewer()

    # Hepatocytes
    viewer.add_labels(hepa_img, name='Voxeles: Hepatocytes', opacity=0.3, visible=False)
    
    # Network Meshes
    #viewer.add_surface((mesh_bile.vertices, mesh_bile.faces),
                         #  name='Malla: Bilis', colormap='yellow', opacity=0.4, blending='translucent')
    
    #viewer.add_surface((mesh_sinu.vertices, mesh_sinu.faces),
                          # name='Malla: Sinusoides', colormap='red', opacity=0.4, blending='translucent')

    # Cell Mesh
    for i, (cid, mesh) in enumerate(cell_meshes.items()):
        if i >= 12: break
        viewer.add_surface((mesh.vertices, mesh.faces),
                           name=f'Cell {cid}', colormap='gray', opacity=0.4, blending='additive')

    # Norms
    if sample_cell_id is None:
        sample_cell_id = list(cell_meshes.keys())[5]
    
    target_mesh = cell_meshes.get(sample_cell_id)
    
    step = 500 
    origins = target_mesh.triangles_center[::step]
    vectors = target_mesh.face_normals[::step]
    
    scale = 25.0
    vector_data = np.zeros((len(origins), 2, 3))
    vector_data[:, 0, :] = origins
    vector_data[:, 1, :] = vectors * scale

    viewer.add_vectors(
        vector_data,
        name='Norms',
        edge_width=1.5,
        edge_color='red',
        length=1.5 
    )
    
    return viewer

# EJECUCIÓN
viewer = visualize_meshes(hepa_img, bile_img, sinu_img, cell_meshes)

In [6]:
def map_network_topology(label_volume, cell_meshes, 
                         start_dist=1.5, end_dist=5.0, step_dist=0.5):
    """
    Performs Ray-Casting to identify the neighbor of each face in the mesh.
    """

    # Image Limits
    max_z, max_y, max_x = label_volume.shape
    
    # Probe distance range
    search_distances = np.arange(start_dist, end_dist + step_dist, step_dist)
    
    # Dictionary to store the results
    mapped_cell_data = {}
    
    # Iterate cell by cell
    for cid, mesh in tqdm(cell_meshes.items()):
        
        # Geometry
        face_centers = mesh.triangles_center
        face_normals = mesh.face_normals
        face_areas = mesh.area_faces
        num_faces = len(face_centers)
        
        # Initial Values
        face_topology_labels = np.zeros(num_faces, dtype=int)
        
        # Boolean mask tracking faces that haven't found a neighbor yet
        is_unresolved = np.ones(num_faces, dtype=bool)
        
        # Iterative Ray Casting
        for dist in search_distances:
            
            # Indices that still need calculation
            current_indices = np.where(is_unresolved)[0]
            
            # Probe coordinates
            current_centers = face_centers[current_indices]
            current_normals = face_normals[current_indices]
            probe_coords = current_centers + (current_normals * dist)
            probe_indices = np.round(probe_coords).astype(int)
            
            # Boundary Check
            is_outside_z = (probe_indices[:, 0] < 5) | (probe_indices[:, 0] >= max_z - 5)
            is_outside_y = (probe_indices[:, 1] < 5) | (probe_indices[:, 1] >= max_y - 5)
            is_outside_x = (probe_indices[:, 2] < 5) | (probe_indices[:, 2] >= max_x - 5)
            
            is_border = is_outside_z | is_outside_y | is_outside_x
            
            # State: Borders
            if np.any(is_border):
                # Indices of faces that hit the border
                border_indices = current_indices[is_border]
                # Assign ID_BORDER
                face_topology_labels[border_indices] = ID_BORDER
                # Mark as resolved
                is_unresolved[border_indices] = False
            
            # Inside the image
            is_inside = ~is_border
            
            if np.any(is_inside):
                # Get coordinates
                valid_coords = probe_indices[is_inside]
                
                # Read the Master Mask
                voxel_ids = label_volume[valid_coords[:, 0], valid_coords[:, 1], valid_coords[:, 2]]
                
                # Valid Neighbours
                is_valid_neighbor = (voxel_ids != 0) & (voxel_ids != cid)
                
                # Update State:
                if np.any(is_valid_neighbor):
                    inside_indices = current_indices[is_inside]
                    found_indices = inside_indices[is_valid_neighbor]
                    # Store the ID found (Bile, Sinu, or Cell ID)
                    found_values = voxel_ids[is_valid_neighbor]
                    face_topology_labels[found_indices] = found_values
                    # Mark as resolved
                    is_unresolved[found_indices] = False
        
        # Calculate Areas
        # Create boolean masks for each category
        mask_apical  = (face_topology_labels == ID_BILE)
        mask_basal   = (face_topology_labels == ID_SINU)
        mask_border  = (face_topology_labels == ID_BORDER)
        mask_void    = (face_topology_labels == 0)
        mask_self    = (face_topology_labels == cid) 
        mask_cv = (face_topology_labels == ID_CV)
        mask_pv = (face_topology_labels == ID_PV)
        mask_lateral = (
            (face_topology_labels > 0)
            & (~mask_apical)
            & (~mask_basal)
            & (~mask_border)
            & (~mask_self)
            & (~mask_cv)
            & (~mask_pv)
        )
        # Compute Areas
        area_apical  = np.sum(face_areas[mask_apical])
        area_basal   = np.sum(face_areas[mask_basal])
        area_lateral = np.sum(face_areas[mask_lateral])
        area_border  = np.sum(face_areas[mask_border])
        area_void    = np.sum(face_areas[mask_void])
        area_self    = np.sum(face_areas[mask_self])
        
        # Save to dictionary
        mapped_cell_data[cid] = {
            'cellId': cid,
            'Volume': mesh.volume,
            'Center_Mass': mesh.center_mass.tolist(),
            'Area_Total': np.sum(face_areas),
            'face_labels': face_topology_labels, 
            'Area_Apical': area_apical,
            'Area_Basal': area_basal,
            'Area_Lateral': area_lateral,
            'Area_Border': area_border,
            'Area_Void': area_void,
            'Area_Self': area_self
        }

    return mapped_cell_data

cell_data = map_network_topology(label_volume=master_labels, cell_meshes=cell_meshes, start_dist=1.0, end_dist=25.0, step_dist=0.5)

100%|█████████████████████████████████████████| 647/647 [00:15<00:00, 41.07it/s]


In [15]:
def visualize_final_verification(cell_data, cell_meshes, hepa_img, bile_img, sinu_img):
    """
    Visualize the identified meshes
    """
    
    # Geometry
    all_vertices = []
    
    # Lists for each type
    faces_bile = []    # Apical
    faces_sinu = []    # Basal
    faces_lateral = [] # Cell-Cell
    faces_border = []  # Outside Box
    faces_void = []    # Unresolved
    faces_original_all = [] # Original Mesh
    
    vertex_offset = 0
    
    # dictionary loop
    for cid, data in cell_data.items():
        if cid not in cell_meshes: continue
        
        mesh = cell_meshes[cid]
        labels = data['face_labels'] # IDs
        
        # Comulate vertices
        all_vertices.append(mesh.vertices)
        
        current_faces = mesh.faces + vertex_offset
        
        faces_original_all.append(current_faces)
        
        # Apical
        mask_bile = (labels == ID_BILE)
        if np.any(mask_bile): faces_bile.append(current_faces[mask_bile])
            
        # Basal
        mask_sinu = (labels == ID_SINU)
        if np.any(mask_sinu): faces_sinu.append(current_faces[mask_sinu])
            
        # Limits
        mask_border = (labels == ID_BORDER)
        if np.any(mask_border): faces_border.append(current_faces[mask_border])
            
        # Void
        mask_void = (labels == 0)
        if np.any(mask_void): faces_void.append(current_faces[mask_void])
            
        # Cell-Cell
        mask_lateral = (labels > 0) & (~mask_bile) & (~mask_sinu) & (~mask_border) & (labels != cid)
        if np.any(mask_lateral): faces_lateral.append(current_faces[mask_lateral])
            
        # Next Cell
        vertex_offset += len(mesh.vertices)

    master_vertices = np.vstack(all_vertices)

    viewer = napari.Viewer()
    
    # RAW Images
    
    # Hepatocytes
    viewer.add_labels(
        hepa_img, 
        name='RAW: Hepatocytes', 
        opacity=0.15, 
        visible=False
    )
    
    # Bile canaliculi
    viewer.add_image(
        bile_img > 0, 
        name='RAW: Bile', 
        colormap='yellow', 
        opacity=0.5, 
        blending='additive',
        visible=False
    )
    
    # Sinusoids
    viewer.add_image(
        sinu_img > 0, 
        name='RAW: Sinusoid', 
        colormap='orange', 
        opacity=0.5, 
        blending='additive',
        visible=False
    )

    # Mesh helper function
    def add_surf(face_list, name, color, opacity=1.0, visible=True):
        if len(face_list) > 0:
            # vstack es necesario para convertir lista de arrays en un solo array
            faces_concat = np.vstack(face_list)
            viewer.add_surface(
                (master_vertices, faces_concat),
                name=f"MESH: {name}",
                colormap=color,
                opacity=opacity,
                visible=visible
            )

    # Void
    add_surf(faces_void, "Void", "red", opacity=1.0, visible=False)
    
    # Apical 
    add_surf(faces_bile, "Apical", "cyan", opacity=1.0, visible=False)
    
    # Sinusoids
    add_surf(faces_sinu, "Basal", "magenta", opacity=0.6, visible=False)
    
    # Cell-Cell
    add_surf(faces_lateral, "Cell-Cell", "green", opacity=0.3, visible=False)

    # Limits
    add_surf(faces_border, "Crop Limit", "gray", opacity=0.5, visible=False)
    
    # Original Mesh
    add_surf(faces_original_all, "Original Mesh", "yellow", opacity=0.3, visible=False)
    
    return viewer

# EXECUTION
viewer = visualize_final_verification(cell_data, cell_meshes, hepa_img, bile_img, sinu_img)

In [7]:
def compute_cell_physics(cell_data, cell_meshes, bile_id):
    """
    Computes nematic tensor (polarity) and inertia tensor (shape)
    for each cell in the dataset, following the methodology of
    Morales-Navarrete et al. / Subía thesis as closely as possible.
    """

    def compute_nematic_angle(vec_a, vec_b):
        cos_theta = np.clip(np.abs(np.dot(vec_a, vec_b)), 0.0, 1.0)
        return np.degrees(np.arccos(cos_theta))

    bile_contact_count = 0

    for cid, data in cell_data.items():
        mesh = cell_meshes[cid]
        face_labels = data['face_labels']

        # -----------------------------
        # DEFAULT OUTPUTS
        # -----------------------------
        Q_tensor = np.zeros((3, 3), dtype=float)
        eigvals_nematic = np.array([np.nan, np.nan, np.nan], dtype=float)

        bipolar_axis = np.zeros(3, dtype=float)
        ring_axis = np.zeros(3, dtype=float)
        middle_axis = np.zeros(3, dtype=float)

        bipolar_weight = np.nan
        ring_weight = np.nan
        middle_weight = np.nan
        S_order = np.nan

        spherical_bile_area = 0.0

        # -----------------------------
        # 1. NEMATIC TENSOR (APICAL)
        # -----------------------------
        bile_indices = np.where(face_labels == bile_id)[0]

        if len(bile_indices) > 0:
            # Use volumetric center consistently
            center_mass = mesh.center_mass

            # Extract apical triangles
            apical_faces = mesh.faces[bile_indices]
            v_original = mesh.vertices[apical_faces]   # shape: (n_tri, 3, 3)

            # Translate to origin
            v_centered = v_original - center_mass

            # Project vertices onto unit sphere
            v_norms = norm(v_centered, axis=2, keepdims=True)
            v_sphere = v_centered / (v_norms + 1e-12)

            # Triangle vertices on sphere
            A = v_sphere[:, 0, :]
            B = v_sphere[:, 1, :]
            C = v_sphere[:, 2, :]

            # Great-circle normals for Girard formula
            N_c = np.cross(A, B)
            N_a = np.cross(B, C)
            N_b = np.cross(C, A)

            N_c /= (norm(N_c, axis=1, keepdims=True) + 1e-12)
            N_a /= (norm(N_a, axis=1, keepdims=True) + 1e-12)
            N_b /= (norm(N_b, axis=1, keepdims=True) + 1e-12)

            # Interior angles
            ang_A = np.arccos(np.clip(-np.sum(N_b * N_c, axis=1), -1.0, 1.0))
            ang_B = np.arccos(np.clip(-np.sum(N_c * N_a, axis=1), -1.0, 1.0))
            ang_C = np.arccos(np.clip(-np.sum(N_a * N_b, axis=1), -1.0, 1.0))

            # Spherical triangle area
            spherical_areas = ang_A + ang_B + ang_C - np.pi

            # Numerical cleanup
            spherical_areas = np.maximum(spherical_areas, 0.0)
            valid = spherical_areas > 1e-10

            if np.any(valid):
                A = A[valid]
                B = B[valid]
                C = C[valid]
                spherical_areas = spherical_areas[valid]

                spherical_bile_area = np.sum(spherical_areas)

                # IMPORTANT:
                # Use triangle normals on the projected sphere,
                # not radial centroid directions
                tri_normals = np.cross(B - A, C - A)
                tri_normals /= (norm(tri_normals, axis=1, keepdims=True) + 1e-12)

                # Build nematic tensor
                Q_accumulator = np.zeros((3, 3), dtype=float)

                for n_i, area_i in zip(tri_normals, spherical_areas):
                    Q_accumulator += area_i * (
                        np.outer(n_i, n_i) - (1.0 / 3.0) * np.eye(3)
                    )

                Q_tensor = (3.0 / 2.0) * (Q_accumulator / spherical_bile_area)

                # Diagonalize
                eigvals, eigvecs = eigh(Q_tensor)  # ascending order

                # Smallest -> ring axis
                # Largest  -> bipolar axis
                ring_axis = eigvecs[:, 0]
                middle_axis = eigvecs[:, 1]
                bipolar_axis = eigvecs[:, 2]

                ring_weight = eigvals[0]
                middle_weight = eigvals[1]
                bipolar_weight = eigvals[2]

                eigvals_nematic = eigvals.copy()
                S_order = bipolar_weight
                bile_contact_count += 1

        # -----------------------------
        # 2. SHAPE TENSOR
        # -----------------------------
        inertia_tensor = mesh.moment_inertia
        shape_vals, shape_vecs = eigh(inertia_tensor)

        shape_director = shape_vecs[:, 0]
        shape_middle = shape_vecs[:, 1]
        shape_third = shape_vecs[:, 2]

        shape_anisotropy = 1.0 - (shape_vals[0] / (shape_vals[2] + 1e-12))

        # -----------------------------
        # 3. COUPLING ANGLES
        # -----------------------------
        if np.sum(np.abs(bipolar_axis)) > 0:
            angle_b1_s1 = compute_nematic_angle(bipolar_axis, shape_director)
            angle_b1_s3 = compute_nematic_angle(bipolar_axis, shape_third)
            angle_b3_s1 = compute_nematic_angle(ring_axis, shape_director)
            angle_b3_s3 = compute_nematic_angle(ring_axis, shape_third)
        else:
            angle_b1_s1 = np.nan
            angle_b1_s3 = np.nan
            angle_b3_s1 = np.nan
            angle_b3_s3 = np.nan

        # -----------------------------
        # 4. SAVE RESULTS
        # -----------------------------
        cell_data[cid]['physics'] = {
            # Nematic tensor
            'Q_tensor': Q_tensor,
            'eigvals_nematic': eigvals_nematic,

            # Nematic axes
            'bipolar_axis': bipolar_axis,
            'ring_axis': ring_axis,
            'middle_axis': middle_axis,

            # Nematic weights
            'bipolar_weight': bipolar_weight,
            'ring_weight': ring_weight,
            'middle_weight': middle_weight,
            'S_order': S_order,

            # QC
            'spherical_bile_area': spherical_bile_area,
            'cell_center_used': mesh.center_mass,

            # Shape
            'shape_director': shape_director,
            'shape_middle': shape_middle,
            'shape_third': shape_third,
            'shape_anisotropy': shape_anisotropy,

            # Coupling
            'angle_b1_s1': angle_b1_s1,
            'angle_b1_s3': angle_b1_s3,
            'angle_b3_s1': angle_b3_s1,
            'angle_b3_s3': angle_b3_s3,
        }

    print(f"Calculation completed for {bile_contact_count} cells with bile contact.")
    return cell_data

cell_data = compute_cell_physics(cell_data, cell_meshes, bile_id=ID_BILE)

Calculation completed for 613 cells with bile contact.


In [8]:
def qc_total_spherical_area(cell_meshes, sample_ids=None):
    """
    Quality control:
    project ALL triangles of a cell to the sphere and check whether
    the total spherical area is close to 4*pi.
    """
    results = []

    if sample_ids is None:
        sample_ids = list(cell_meshes.keys())

    for cid in sample_ids:
        mesh = cell_meshes[cid]
        center = mesh.center_mass

        faces = mesh.faces
        v = mesh.vertices[faces]
        v_centered = v - center

        v_norms = norm(v_centered, axis=2, keepdims=True)
        v_sphere = v_centered / (v_norms + 1e-12)

        A = v_sphere[:, 0, :]
        B = v_sphere[:, 1, :]
        C = v_sphere[:, 2, :]

        N_c = np.cross(A, B)
        N_a = np.cross(B, C)
        N_b = np.cross(C, A)

        N_c /= (norm(N_c, axis=1, keepdims=True) + 1e-12)
        N_a /= (norm(N_a, axis=1, keepdims=True) + 1e-12)
        N_b /= (norm(N_b, axis=1, keepdims=True) + 1e-12)

        ang_A = np.arccos(np.clip(-np.sum(N_b * N_c, axis=1), -1.0, 1.0))
        ang_B = np.arccos(np.clip(-np.sum(N_c * N_a, axis=1), -1.0, 1.0))
        ang_C = np.arccos(np.clip(-np.sum(N_a * N_b, axis=1), -1.0, 1.0))

        spherical_areas = ang_A + ang_B + ang_C - np.pi
        spherical_areas = np.maximum(spherical_areas, 0.0)

        total_area = np.sum(spherical_areas)

        results.append({
            "cell_id": cid,
            "total_spherical_area": total_area,
            "ratio_to_4pi": total_area / (4 * np.pi)
        })

    return pd.DataFrame(results)

In [9]:
df_qc = qc_total_spherical_area(cell_meshes)
print(df_qc["ratio_to_4pi"].describe())

count    647.000000
mean       1.010536
std        0.045837
min        0.957222
25%        1.002342
50%        1.004337
75%        1.008594
max        1.876977
Name: ratio_to_4pi, dtype: float64


In [10]:
def get_valid_cells_for_nematic_averaging(
    cell_data,
    qc_df=None,
    max_border_fraction=0.05,
    qc_ratio_min=0.90,
    qc_ratio_max=1.20
):
    """
    Select cells suitable for nematic averaging.
    """
    valid_ids = []

    qc_map = None
    if qc_df is not None:
        qc_map = dict(zip(qc_df["cell_id"], qc_df["ratio_to_4pi"]))

    for cid, data in cell_data.items():
        # Must have physics
        if 'physics' not in data:
            continue

        phys = data['physics']

        # Must have valid bipolar axis
        axis = phys.get('bipolar_axis', None)
        if axis is None:
            continue
        if not np.all(np.isfinite(axis)):
            continue
        if np.linalg.norm(axis) < 1e-12:
            continue

        # Border filter
        area_total = data.get('Area_Total', 0.0)
        area_border = data.get('Area_Border', 0.0)
        if area_total <= 0:
            continue

        border_fraction = area_border / area_total
        if border_fraction > max_border_fraction:
            continue

        # QC filter if available
        if qc_map is not None and cid in qc_map:
            ratio = qc_map[cid]
            if not (qc_ratio_min <= ratio <= qc_ratio_max):
                continue

        valid_ids.append(cid)

    return valid_ids

In [11]:
df_qc = qc_total_spherical_area(cell_meshes)

valid_ids = get_valid_cells_for_nematic_averaging(
    cell_data,
    qc_df=df_qc,
    max_border_fraction=0.05,
    qc_ratio_min=0.90,
    qc_ratio_max=1.20
)

print("Valid cells for averaging:", len(valid_ids))

Valid cells for averaging: 224


In [12]:
def compute_local_averaged_nematic_field(
    cell_data,
    valid_ids,
    axis_key='bipolar_axis',
    sigma_um=20.0,
    voxel_size_um=(1.0, 1.0, 1.0),
    normalize_weights=True,
    min_effective_weight=1e-6
):
    """
    Compute local Gaussian coarse-grained nematic tensor field at cell centers,
    following the methodology of Morales-Navarrete et al.

    Parameters
    ----------
    cell_data : dict
        Dictionary with per-cell information.
    valid_ids : list
        Cell IDs to include in the averaging.
    axis_key : str
        Which nematic axis to average. Examples:
        - 'bipolar_axis'
        - 'ring_axis'
    sigma_um : float
        Gaussian kernel std in micrometers. Paper uses 20 µm.
    voxel_size_um : tuple
        Physical voxel size in (z, y, x) micrometers.
    normalize_weights : bool
        If True, divide by sum of weights. This does not change eigenvectors,
        but improves numerical stability.
    min_effective_weight : float
        Minimum total weight to trust the local estimate.

    Returns
    -------
    field_data : dict
        Per-cell local averaged nematic tensor and principal axis.
    """
    voxel_size_um = np.asarray(voxel_size_um, dtype=float)

    # Gather centers and axes
    centers = []
    axes = []
    ids = []

    for cid in valid_ids:
        data = cell_data[cid]
        phys = data['physics']

        c = np.asarray(data['Center_Mass'], dtype=float)
        a = np.asarray(phys[axis_key], dtype=float)

        # normalize axis just in case
        a_norm = np.linalg.norm(a)
        if a_norm < 1e-12:
            continue
        a = a / a_norm

        centers.append(c)
        axes.append(a)
        ids.append(cid)

    centers = np.asarray(centers, dtype=float)   # (N, 3)
    axes = np.asarray(axes, dtype=float)         # (N, 3)

    if len(ids) == 0:
        return {}

    # Convert sigma from µm to voxel units per axis
    sigma_vox = sigma_um / voxel_size_um   # (3,)

    field_data = {}

    # Evaluate coarse-grained tensor at each cell center xi
    for i, cid_eval in enumerate(ids):
        x_i = centers[i]

        # anisotropic Gaussian distance in physical units
        delta = centers - x_i                      # (N, 3) in voxels
        delta_um = delta * voxel_size_um[None, :] # convert to µm

        sq = np.sum((delta_um / sigma_um) ** 2, axis=1)
        weights = np.exp(-0.5 * sq)

        # Build local weighted tensor
        N_local = np.zeros((3, 3), dtype=float)

        for w, a in zip(weights, axes):
            N_local += w * (np.outer(a, a) - np.eye(3) / 3.0)

        N_local *= 3.0 / 2.0

        weight_sum = np.sum(weights)

        if normalize_weights and weight_sum > min_effective_weight:
            N_local /= weight_sum

        # Diagonalize
        eigvals, eigvecs = eigh(N_local)  # ascending
        principal_axis = eigvecs[:, 2]    # largest eigenvalue
        principal_value = eigvals[2]

        field_data[cid_eval] = {
            'center': x_i,
            'N_local': N_local,
            'eigvals_local': eigvals,
            'principal_axis': principal_axis,
            'principal_value': principal_value,
            'weight_sum': weight_sum,
            'axis_key': axis_key,
            'sigma_um': sigma_um,
        }

    return field_data

In [13]:
local_bipolar_field = compute_local_averaged_nematic_field(
    cell_data=cell_data,
    valid_ids=valid_ids,
    axis_key='bipolar_axis',
    sigma_um=20.0,
    voxel_size_um=(1.0, 1.0, 1.0),   # cambia esto si tus voxeles no son 1 µm
    normalize_weights=True
)

In [14]:
local_ring_field = compute_local_averaged_nematic_field(
    cell_data=cell_data,
    valid_ids=valid_ids,
    axis_key='ring_axis',
    sigma_um=20.0,
    voxel_size_um=(1.0, 1.0, 1.0),
    normalize_weights=True
)

In [15]:
def visualize_local_nematic_field(
    local_field,
    hepa_img=None,
    scale_vectors=20.0,
    layer_name="Local Nematic Field",
    points_name="Cell Centers",
    vector_color="cyan",
    show_points=True
):
    """
    Visualize local coarse-grained nematic axes in Napari.
    """
    viewer = napari.Viewer(ndisplay=3)

    if hepa_img is not None:
        viewer.add_labels(
            hepa_img,
            name="Hepatocytes",
            opacity=0.10,
            visible=False
        )

    centers = []
    vector_data = []

    for cid, d in local_field.items():
        c = np.asarray(d['center'], dtype=float)
        a = np.asarray(d['principal_axis'], dtype=float)

        if not np.all(np.isfinite(a)):
            continue
        if np.linalg.norm(a) < 1e-12:
            continue

        centers.append(c)
        vector_data.append([c, a * scale_vectors])

    centers = np.asarray(centers)
    vector_data = np.asarray(vector_data)

    if show_points and len(centers) > 0:
        viewer.add_points(
            centers,
            size=5,
            face_color='white',
            opacity=0.7,
            name=points_name
        )

    if len(vector_data) > 0:
        viewer.add_vectors(
            vector_data,
            edge_width=1.0,
            length=1.0,
            edge_color=vector_color,
            name=layer_name
        )

    return viewer

In [20]:
viewer = visualize_local_nematic_field(
    local_bipolar_field,
    hepa_img=hepa_img,
    scale_vectors=20.0,
    layer_name="Local Averaged Bipolar Axis",
    vector_color="cyan"
)

In [17]:
viewer = visualize_local_nematic_field(
    local_ring_field,
    hepa_img=hepa_img,
    scale_vectors=20.0,
    layer_name="Local Averaged Ring Axis",
    vector_color="blue"
)

In [18]:
def build_tissue_graph(cell_data, cell_meshes, 
                       id_bile, id_sinu, id_border):
    """
    Constructs graph representing the tissue architecture
    Nodes: Hepatocytes
    Edges: Physical contacts between cells
    """

    # Initialize an empty undirected graph
    G = nx.Graph()
    
    # Iterate through every cell in our master dictionary
    for cid, data in cell_data.items():
        
        # CREATE NODE (Hepatocyte)
        
        # Geometry
        attributes = {
            'cell_id': cid,
            'volume': data['Volume'],
            'center_mass': np.array(data['Center_Mass']),
            # Total Areas (Scalar values)
            'area_apical': data['Area_Apical'],
            'area_basal': data['Area_Basal'],
            'area_lateral': data['Area_Lateral']
        }
        
        # Physics Attributes
        if 'physics' in data:
            phys = data['physics']
            
            # Scalars
            attributes['s_order'] = phys['S_order']
            attributes['shape_anisotropy'] = phys['shape_anisotropy']
            attributes['spherical_bile_area'] = phys.get('spherical_bile_area', 0.0)

            # Coupling Angles
            attributes['angle_b1_s1'] = phys['angle_b1_s1']
            attributes['angle_b1_s3'] = phys['angle_b1_s3']
            attributes['angle_b3_s1'] = phys['angle_b3_s1']
            attributes['angle_b3_s3'] = phys['angle_b3_s3']
            
            # Vectors
            attributes['vec_bipolar_axis'] = phys['bipolar_axis']
            attributes['vec_ring_axis'] = phys['ring_axis']
            attributes['vec_shape_director'] = phys['shape_director']
            attributes['vec_shape_third'] = phys['shape_third']
            
            attributes['bipolar_weight'] = phys['bipolar_weight']
            attributes['ring_weight'] = phys['ring_weight']
            attributes['eigvals_nematic'] = phys['eigvals_nematic']
        # Add the node to the graph with all attributes
        G.add_node(cid, **attributes)

        # CREATE EDGES
        
        # Find Neighbours
        face_labels = data['face_labels']
        if cid in cell_meshes:
            mesh_areas = cell_meshes[cid].area_faces
            
            # neighbors touching this cell
            unique_neighbors = np.unique(face_labels)
            
            for neighbor_id in unique_neighbors:
                
                # FILTER: Ignore non-cellular structures
                if neighbor_id == 0: continue
                if neighbor_id == cid: continue
                if neighbor_id == id_bile: continue
                if neighbor_id == id_sinu: continue
                if neighbor_id == id_border: continue
                
                # Calculate Shared Area
                contact_indices = np.where(face_labels == neighbor_id)[0]
                shared_area = np.sum(mesh_areas[contact_indices])
                
                # Calculate Distance between Centroids
                if neighbor_id in cell_data:
                    pos_a = np.array(data['Center_Mass'])
                    pos_b = np.array(cell_data[neighbor_id]['Center_Mass'])
                    
                    # Euclidean distance
                    dist_centroids = distance.euclidean(pos_a, pos_b)
                    
                    # Add the Edge to the Graph
                    G.add_edge(cid, neighbor_id, 
                               weight_area=shared_area, 
                               weight_distance=dist_centroids)
                

    print(f" - Nodes (Cells): {G.number_of_nodes()}")
    print(f" - Edges (Contacts): {G.number_of_edges()}")
    
    return G
tissue_graph = build_tissue_graph(cell_data, cell_meshes, id_bile=ID_BILE, id_sinu=ID_SINU, id_border=ID_BORDER)

 - Nodes (Cells): 647
 - Edges (Contacts): 2925


In [19]:
def visualize_tissue_graph(graph, scale_vectors=15.0):
    """
    Visualizes the Tissue Graph in Napari.
    """
    node_ids = list(graph.nodes())
    node_positions = []
    
    for nid in node_ids:
        pos = graph.nodes[nid]['center_mass'] # It's already a numpy array or list
        node_positions.append(pos)
        
    node_positions = np.array(node_positions)
    
    edge_lines = []
    
    for u, v in graph.edges():
        pos_u = graph.nodes[u]['center_mass']
        pos_v = graph.nodes[v]['center_mass']
        edge_lines.append([pos_u, pos_v])
        
    edge_lines = np.array(edge_lines)
    
    
    def extract_vectors(attr_name):
        vector_data = []
        for nid in node_ids:
            # Check if physics exists for this node
            if attr_name in graph.nodes[nid]:
                origin = graph.nodes[nid]['center_mass']
                vector = graph.nodes[nid][attr_name]
                
                # Check for NaNs or Zeros (e.g. border cells)
                if np.sum(np.abs(vector)) > 0:
                    # Apply visual scaling so we can see them
                    scaled_vector = vector * scale_vectors
                    vector_data.append([origin, scaled_vector])
        return np.array(vector_data)

    # Extract the 4 requested vectors
    vec_bile_1 = extract_vectors('vec_bipolar_axis')
    vec_bile_3 = extract_vectors('vec_ring_axis')
        
    vec_shape_1 = extract_vectors('vec_shape_director')
    vec_shape_3 = extract_vectors('vec_shape_third')


    viewer = napari.Viewer()
    
    # A. Add Edges (Bottom Layer)
    if len(edge_lines) > 0:
        viewer.add_shapes(
            edge_lines,
            shape_type='line',
            edge_width=0.5,
            edge_color='yellow',
            opacity=0.4,
            name='Graph Edges (Contacts)'
        )
        
    # B. Add Nodes
    if len(node_positions) > 0:
        viewer.add_points(
            node_positions,
            size=6,
            face_color='white',
            opacity=0.8,
            name='Graph Nodes (Centroids)'
        )

    # C. Add Physics Vectors
    # Note: We toggle visibility to avoid clutter
    
    # Shape Vectors (Magenta/Purple)
    if len(vec_shape_1) > 0:
        viewer.add_vectors(
            vec_shape_1,
            edge_width=0.8,
            length=1, # Length is already scaled in data
            edge_color='magenta',
            name='Shape: Director (1st)',
            visible=False 
        )
        
    if len(vec_shape_3) > 0:
        viewer.add_vectors(
            vec_shape_3,
            edge_width=0.5,
            length=1,
            edge_color='purple',
            name='Shape: Minor (3rd)',
            visible=False
        )

    # Bile Vectors (Cyan/Blue) - The most important ones
    if len(vec_bile_1) > 0:
        viewer.add_vectors(
            vec_bile_1,
            edge_width=1.0, # Thicker
            length=1,
            edge_color='cyan',
            name='Nematic: Bipolar Axis',
            visible=True
        )

    if len(vec_bile_3) > 0:
        viewer.add_vectors(
            vec_bile_3,
            edge_width=0.5,
            length=1,
            edge_color='blue',
            name='Nematic: Ring Axis',
            visible=False
        )

    return viewer


viewer = visualize_tissue_graph(tissue_graph, scale_vectors=20.0)

In [ ]:
# Compute skeleton for bile canaliculi
bile_binary = bile_img > 0
bile_skeleton = skeletonize(bile_binary)

print(f"Original voxels: {np.sum(bile_binary)}")
print(f"Skeleton voxels: {np.sum(bile_skeleton)}")

In [ ]:
# Compute skeleton for sinusoids
sinu_binary = sinu_img > 0
sinu_skeleton = skeletonize(sinu_binary)

print(f"Original voxels: {np.sum(sinu_binary)}")
print(f"Skeleton voxels: {np.sum(sinu_skeleton)}")

In [38]:
# Convert sinosoid binary mask to labeled volume
labels_sinu = np.zeros_like(sinu_img, dtype=np.uint32)
labels_sinu[sinu_img > 0] = 1

print("Unique labels:", np.unique(labels_sinu))

Unique labels: [0 1]


In [51]:
skels = kimimaro.skeletonize(
    labels_sinu,

    teasar_params={
        "scale": 3.0,
        "const": 20,
        "pdrf_scale": 100000,
        "pdrf_exponent": 4,
        #"max_paths": 1e3,
        "max_paths": None,
    },

    dust_threshold=500,
    anisotropy=(1, 1, 1),
    fix_branching=True,
    fix_borders=True,
    fill_holes=False,

    progress=True,
    parallel=14
)

print("Skeletons computed:", len(skels))

Skeletonizing Labels: 100%|███████████████████████| 3/3 [01:05<00:00, 21.71s/it]

Skeletons computed: 1


In [52]:
skel = skels[1]

vertices = skel.vertices  # (N, 3)
edges = skel.edges        # (M, 2)

print("Vertices:", vertices.shape)
print("Edges:", edges.shape)

Vertices: (16660, 3)
Edges: (16657, 2)


In [53]:
def visualize_kimimaro_skeleton(vertices, edges, raw_mask=None, raw_name="Raw Image"):
    viewer = napari.Viewer(ndisplay=3)

    if raw_mask is not None:
        viewer.add_labels(
            raw_mask > 0,
            name=raw_name,
            opacity=0.8
        )

    lines = []
    for e in edges:
        p1 = vertices[e[0]]
        p2 = vertices[e[1]]
        lines.append([p1, p2])
    lines = np.array(lines)

    viewer.add_shapes(
        lines,
        shape_type='line',
        edge_width=1,
        edge_color='cyan',
        name="Kimimaro Skeleton"
    )

    return viewer

In [54]:
viewer = visualize_kimimaro_skeleton(vertices, edges, sinu_img, raw_name="Raw Sinusoids")

In [5]:
# Convert bile canaliculi binary mask to labeled volume
labels_bile = np.zeros_like(bile_img, dtype=np.uint32)
labels_bile[bile_img > 0] = 1

print("Unique labels:", np.unique(labels_bile))

Unique labels: [0 1]


In [35]:
skels = kimimaro.skeletonize(
    labels_bile,

    teasar_params={
        "scale": 3.0,
        "const": 15,
        "pdrf_scale": 100000,
        "pdrf_exponent": 4,
        "max_paths": 400,
        #"max_paths": None,
    },

    dust_threshold=700,
    anisotropy=(1, 1, 1),
    fix_branching=True,
    fix_borders=True,
    fill_holes=False,

    progress=True,
    parallel=14
)

print("Skeletons computed:", len(skels))

Skeletonizing Labels:  52%|██████████▉          | 14/27 [01:02<00:57,  4.46s/it]

Skeletons computed: 1


In [36]:
skel = skels[1]

vertices = skel.vertices  # (N, 3)
edges = skel.edges        # (M, 2)

print("Vertices:", vertices.shape)
print("Edges:", edges.shape)

Vertices: (20160, 3)
Edges: (20133, 2)


In [37]:
viewer = visualize_kimimaro_skeleton(vertices, edges, bile_img, raw_name="Raw Bile")

In [ ]:
def detect_junctions(skeleton):
    """
    Detect junction voxels in a 3D skeleton.
    """
    skel = skeleton.astype(bool)

    # 3x3x3 kernel to count neighbors
    kernel = np.ones((3, 3, 3), dtype=np.uint8)

    # Count skeleton voxels in the neighborhood
    neighborhood_sum = ndi.convolve(
        skel.astype(np.uint8),
        kernel,
        mode='constant',
        cval=0
    )

    # Subtract the center voxel
    neighbor_count = neighborhood_sum - skel.astype(np.uint8)

    # Junction candidates
    junction_mask = skel & (neighbor_count > 2)

    # Label connected junction regions
    structure = np.ones((3, 3, 3), dtype=np.uint8)
    junction_labels, num_junctions = ndi.label(junction_mask, structure=structure)

    # Store junction info
    junction_data = {}

    # Find minimal bounding boxes for each labeled cluster
    objects = ndi.find_objects(junction_labels)

    for jid in range(1, num_junctions + 1):
        slc = objects[jid - 1]

        if slc is None:
            continue

        # Work only inside the bounding box of this junction
        local_mask = (junction_labels[slc] == jid)
        local_coords = np.argwhere(local_mask)

        # Convert local coords to global coords
        offset = np.array([
            slc[0].start,
            slc[1].start,
            slc[2].start
        ])
        coords = local_coords + offset

        # Centroid of the cluster
        centroid = coords.mean(axis=0)

        junction_data[jid] = {
            "junction_id": jid,
            "n_voxels": len(coords),
            "centroid": centroid,
            "coords": coords
        }

    return neighbor_count, junction_mask, junction_labels, junction_data

In [ ]:
bile_neighbor_count, bile_junction_mask, bile_junction_labels, bile_junction_data = detect_junctions(bile_skeleton)

print(f"Junction voxels: {np.sum(bile_junction_mask)}")
print(f"Number of junction clusters: {len(bile_junction_data)}")

In [ ]:
def visualize_junctions_on_skeleton(skeleton, junction_mask, junction_data, step=1):
    viewer = napari.Viewer(ndisplay=3)

    # Skeleton as labels
    viewer.add_labels(
        skeleton.astype(np.uint8),
        name="Skeleton",
        opacity=0.25
    )

    # Junction voxels as labels
    viewer.add_labels(
        junction_mask.astype(np.uint8),
        name="Junction Voxels",
        opacity=1.0
    )

    # Junction centroids as points
    if len(junction_data) > 0:
        centroids = np.array([j["centroid"] for j in junction_data.values()])
        sizes = np.array([max(4, min(20, j["n_voxels"])) for j in junction_data.values()])

        viewer.add_points(
            centroids,
            size=sizes,
            face_color="red",
            name="Junction Centroids"
        )

    return viewer

In [ ]:
viewer = visualize_junctions_on_skeleton(
    bile_skeleton,
    bile_junction_mask,
    bile_junction_data
)

In [ ]:
labels_cc, num_cc = ndi.label(bile_img > 0)

print("Connected components in BC:", num_cc)

In [ ]:
def extract_branches_from_skeleton(skeleton, junction_mask):
    """
    Remove junction voxels from the skeleton and label the remaining
    connected branch segments.
    """
    skel = skeleton.astype(bool)
    jmask = junction_mask.astype(bool)

    # Keep only non-junction skeleton voxels
    branch_mask = skel & (~jmask)

    # Label connected branch components
    structure = np.ones((3, 3, 3), dtype=np.uint8) 
    branch_labels, num_branches = ndi.label(branch_mask, structure=structure)

    branch_data = {}

    objects = ndi.find_objects(branch_labels)

    for bid in range(1, num_branches + 1):
        slc = objects[bid - 1]

        if slc is None:
            continue

        local_mask = (branch_labels[slc] == bid)
        local_coords = np.argwhere(local_mask)

        offset = np.array([
            slc[0].start,
            slc[1].start,
            slc[2].start
        ])
        coords = local_coords + offset

        centroid = coords.mean(axis=0)

        branch_data[bid] = {
            "branch_id": bid,
            "n_voxels": len(coords),
            "centroid": centroid,
            "coords": coords
        }

    return branch_mask, branch_labels, branch_data

In [ ]:
bile_branch_mask, bile_branch_labels, bile_branch_data = extract_branches_from_skeleton(
    bile_skeleton,
    bile_junction_mask
)

print(f"Number of canaliculi branch clusters: {len(bile_branch_data)}")

In [ ]:
def visualize_bile_junctions_and_branches(skeleton, junction_mask, branch_mask):
    viewer = napari.Viewer(ndisplay=3)

    viewer.add_labels(
        skeleton.astype(np.uint8),
        name="Bile Skeleton",
        opacity=0.15
    )

    viewer.add_labels(
        junction_mask.astype(np.uint8),
        name="Bile Junctions",
        opacity=1.0
    )

    viewer.add_labels(
        branch_mask.astype(np.uint8),
        name="Bile Branches",
        opacity=0.6
    )

    return viewer

In [ ]:
viewer = visualize_bile_junctions_and_branches(
    bile_skeleton,
    bile_junction_mask,
    bile_branch_mask
)

In [ ]:
sinu_neighbor_count, sinu_junction_mask, sinu_junction_labels, sinu_junction_data = detect_junctions(sinu_skeleton)

print(f"Junction voxels: {np.sum(sinu_junction_mask)}")
print(f"Number of junction clusters: {len(sinu_junction_data)}")

In [ ]:
sinu_branch_mask, sinu_branch_labels, sinu_branch_data = extract_branches_from_skeleton(
    sinu_skeleton,
    sinu_junction_mask
)

print(f"Number of sinusoids branch clusters: {len(sinu_branch_data)}")

In [ ]:
def visualize_sin_junctions_and_branches(skeleton, junction_mask, branch_mask):
    viewer = napari.Viewer(ndisplay=3)

    viewer.add_labels(
        skeleton.astype(np.uint8),
        name="Sinusoid Skeleton",
        opacity=0.15
    )

    viewer.add_labels(
        junction_mask.astype(np.uint8),
        name="Sinusoid Junctions",
        opacity=1.0
    )

    viewer.add_labels(
        branch_mask.astype(np.uint8),
        name="Sinusoid Branches",
        opacity=0.6
    )

    return viewer

In [ ]:
viewer = visualize_sin_junctions_and_branches(
    sinu_skeleton,
    sinu_junction_mask,
    sinu_branch_mask
)

In [ ]:
viewer = visualize_junctions_on_skeleton(
    sinu_skeleton,
    sinu_junction_mask,
    sinu_junction_data
)